# Exploratory ML for Splice Sites

This notebook is designed to implement the SpliceSeqExtractor class in a machine learning context. 

In [1]:
import numpy as np
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent / 'src'))
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from Bio import SeqIO
from extract_splice_seqs import SpliceSeqExtractor, ExtractionParams
from feature_extractor import FeatureExtractor, SpliceModel


## Setup

First, let's define all the file paths that are going to be used throughout the model training and testing steps.

In [2]:
GFF_PATH_TRAIN = "../chr1.gff3"
FASTA_PATH_TRAIN = "../chr1.fa"
GFF_PATH_TEST = "../chr2.gff3"
FASTA_PATH_TEST = "../chr2.fa"
OUT_POS_TRAIN = "chr1pos.fa"
OUT_NEG_TRAIN = "chr1neg.fa"
OUT_POS_TEST = "chr2pos.fa"
OUT_NEG_TEST = "chr2neg.fa"

### About the SpliceSeqExtractor Class and Parameter Choices

Using the SpliceSeqExtractor class, we'll find all splice sites on chromosome 1 for training. I've opted to filter these sequences out by only including protein coding regions, as they tend to be well-annotated, and it will still leave us with plenty of data to train on. Additionally, by default, ExtractionParams will set n_exon = 40, n_intron = 80, and buffer_size = 50. 

This class will find sequences along splice sites given the above parameters. It will output 2 files, one of which contains true splice sites, and for "false" splice sites, it will randomly sample intronic regions, since these by definition cannot be splice sites.

About the parameters, n_exon refers to the number of bases to include in the exon region, and n_intron is the number of bases to include in the intron region. So for the default parameters, we will end up with sequences of length 120, 40 bases of which are inside the exon region, and 80 bases in the intron region. 

Finally, buffer_size refers to the number of bases to exclude from the intron-exon boundaries. Since negative samples are taken from randomly sampling introns, we want to avoid capturing information near known splice sites. Setting this value too high ends up discarding too many sequences, but too low risks capturing information directly near the splice boundary. I may end up tweaking the buffer size and retraining the model as necessary.

Also, as I'm writing this, I'm realizing that I probably should also randomly sample exon regions as well, since the dataset is currently skewed towards positive. This is mainly a result of the buffer size discarding sequences, but adding in exonic regions to the dataset might improve performance. Additionally, I'll probably also add intergenic regions as well.

In [3]:
extractor = SpliceSeqExtractor(gff_path=GFF_PATH_TRAIN, fasta_path=FASTA_PATH_TRAIN, transcript_filter="protein_coding", params=ExtractionParams)
extractor.extract_sequences(positive_output=OUT_POS_TRAIN, negative_output=OUT_NEG_TRAIN)

2025-07-11 14:01:39,335 - INFO - Parsing transcripts from GFF file...
2025-07-11 14:01:41,041 - INFO - Found 186320 transcripts with biotype info
2025-07-11 14:01:45,247 - INFO - Identifying splice junctions...
2025-07-11 14:01:46,256 - INFO - Extracting positive samples to chr1pos.fa...
2025-07-11 14:01:46,950 - INFO - Finding intron coordinates...
2025-07-11 14:01:47,041 - INFO - Extracting negative samples to chr1neg.fa...
2025-07-11 14:01:47,776 - INFO - Extraction complete -- found 238572 positive samples and 103948 negative samples.


(238572, 103948)

### Feature Extraction from DNA Sequences

Additionally, I created a class FeatureExtractor that has methods for obtaining relevant information from DNA sequences. These data include kmer counts, percent composition of each base, and one-hot encoded sequences.

Something to note is that one-hot encoded raw sequence information takes up quite a large amount of memory, and it's unclear whether or not including this is actually useful for traditional ML models. Something I want to explore is whether I should entirely ditch the onehot sequences, or certain positions, since the number of features onehot creates is 4* the length of the sequence (480 in our case). Using regularization should help prevent overfitting from bases positions that aren't as important though.

In [4]:
featextractor = FeatureExtractor(kmer_k=3, flatten_one_hot=True)
sequences_train, labels_train = featextractor.load_data(pos_fasta=OUT_POS_TRAIN, neg_fasta=OUT_NEG_TRAIN)
features_train = featextractor.batch_extract_all(sequences_train)

## Sanity Checks

In [5]:
features_train.shape # correct num of rows

(342520, 548)

## Model Training

In [11]:
params = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "liblinear",
    "max_iter": 1000,
    "class_weight": "balanced"
}

model = SpliceModel(test_size=0.2, random_state=100, model_cls=LogisticRegression, model_params=params)
X_train, X_test, y_train, y_test = model.train_model(features_train, labels_train)
results = model.evaluate(X_test, y_test, metrics=['accuracy', 'roc_auc'])
results

/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_s

{'accuracy': 0.9000788275137218, 'roc_auc': np.float64(0.9511962258545738)}

## Testing the Model

To test the model, we'll use a different chromosome from the human genome.

In [7]:
extractor = SpliceSeqExtractor(gff_path=GFF_PATH_TEST, fasta_path=FASTA_PATH_TEST, transcript_filter="protein_coding", params=ExtractionParams)
extractor.extract_sequences(OUT_POS_TEST, OUT_NEG_TEST)
sequences_test, labels_test = featextractor.load_data(OUT_POS_TEST, OUT_NEG_TEST)
features_test = featextractor.batch_extract_all(sequences_test)

2025-07-11 14:02:57,632 - INFO - Parsing transcripts from GFF file...
2025-07-11 14:02:58,111 - INFO - Found 52514 transcripts with biotype info
2025-07-11 14:02:59,260 - INFO - Identifying splice junctions...
2025-07-11 14:02:59,539 - INFO - Extracting positive samples to chr2pos.fa...
2025-07-11 14:03:00,103 - INFO - Finding intron coordinates...
2025-07-11 14:03:00,126 - INFO - Extracting negative samples to chr2neg.fa...
2025-07-11 14:03:00,506 - INFO - Extraction complete -- found 196146 positive samples and 86791 negative samples.


In [9]:
model.evaluate(features_test, labels_test, metrics=["accuracy", "roc_auc"])

/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_sites_protists/venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/michaelbambha/Documents/projects/splice_s

{'accuracy': 0.9038054407871717, 'roc_auc': np.float64(0.9531872521508385)}